# exp129_spatial_prior_as_selector_candidate train

Train-side audit that adds exp114 fold-safe spatial prior TVT paths to the exp099/101 candidate selector surface.

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, get_nested, load_config
from spatial_prior_as_selector_candidate import (
    DEFAULT_EXP099_CACHE,
    DEFAULT_EXP099_SCHEMA,
    DEFAULT_EXP114_OOF,
    DEFAULT_EXP114_SUMMARY,
    build_required_columns,
    candidate_specs_from_config,
    find_artifact,
    run_spatial_prior_as_selector_candidate,
)

paths = ExperimentPaths()
config = load_config()
output_dir = paths.artifacts_dir
output_dir.mkdir(parents=True, exist_ok=True)

print('experiment:', get_nested(config, 'experiment.name'))
print('route:', get_nested(config, 'experiment.route'))
print('parent:', get_nested(config, 'lineage.parent'))
print('spatial prior parent:', get_nested(config, 'lineage.spatial_prior_parent'))
print('strategy:', get_nested(config, 'validation.strategy'))
print('enable_gpu:', get_nested(config, 'runtime.kaggle.enable_gpu'))
print('output_dir:', output_dir)

## 2. Input artifact audit

In [ ]:
candidates = candidate_specs_from_config(config)
required = build_required_columns(config, candidates)

exp099_cache = find_artifact(
    DEFAULT_EXP099_CACHE,
    get_nested(config, 'data.exp099_train_feature_cache_local'),
    parent_parts=('experiments', 'exp099_pf_multi_observation_likelihood_probe', 'kaggle', 'output', 'train_v2', 'artifacts'),
)
exp099_schema = find_artifact(
    DEFAULT_EXP099_SCHEMA,
    get_nested(config, 'data.exp099_train_feature_schema_local'),
    parent_parts=('experiments', 'exp099_pf_multi_observation_likelihood_probe', 'kaggle', 'output', 'train_v2', 'artifacts'),
)
exp114_oof = find_artifact(
    DEFAULT_EXP114_OOF,
    get_nested(config, 'data.exp114_oof_predictions_local'),
    parent_parts=('experiments', 'exp114_spatial_neighbor_prior_signal_audit', 'kaggle', 'output', 'train_v1', 'artifacts'),
)
exp114_summary = find_artifact(
    DEFAULT_EXP114_SUMMARY,
    get_nested(config, 'data.exp114_summary_local'),
    parent_parts=('experiments', 'exp114_spatial_neighbor_prior_signal_audit', 'kaggle', 'output', 'train_v1', 'artifacts'),
)

exp099_header = pd.read_csv(exp099_cache, nrows=0).columns.tolist()
exp114_header = pd.read_csv(exp114_oof, nrows=0).columns.tolist()
missing_exp099 = [column for column in required['exp099'] if column not in exp099_header]
missing_exp114 = [column for column in required['exp114'] if column not in exp114_header]

print('exp099_cache:', exp099_cache)
print('exp099_schema:', exp099_schema)
print('exp114_oof:', exp114_oof)
print('exp114_summary:', exp114_summary)
print('required exp099 columns:', len(required['exp099']))
print('required exp114 columns:', len(required['exp114']))
print('missing_exp099:', missing_exp099)
print('missing_exp114:', missing_exp114)
if missing_exp099 or missing_exp114:
    raise RuntimeError({'missing_exp099': missing_exp099, 'missing_exp114': missing_exp114})

## 3. Candidate and selector plan

In [ ]:
candidate_plan = pd.DataFrame([
    {
        'candidate': spec.name,
        'column': spec.column,
        'family': spec.family,
        'source_variant': spec.source_variant,
    }
    for spec in candidates
])
selector_plan = pd.DataFrame([
    {'variant': 'likpf_mean_single', 'kind': 'baseline'},
    {'variant': 'oracle_base_only', 'kind': 'base_candidate_upper_bound'},
    {'variant': 'oracle_spatial_only', 'kind': 'spatial_candidate_upper_bound'},
    {'variant': 'oracle_expanded', 'kind': 'expanded_candidate_upper_bound'},
    {'variant': 'lgb_error_ranker_rowwise', 'kind': 'supervised_candidate_long_error_ranker'},
    {'variant': 'lgb_error_ranker_viterbi_*', 'kind': 'viterbi_smoothed_discrete_selector'},
])
display(candidate_plan)
display(selector_plan)
print('n_folds:', get_nested(config, 'validation.n_folds'))
print('long max train rows per fold:', get_nested(config, 'selector.long_model.max_train_rows_per_fold'))
print('viterbi penalties:', get_nested(config, 'selector.viterbi_switch_penalties'))

## 4. Run expanded candidate selector audit

In [ ]:
summary = run_spatial_prior_as_selector_candidate(
    output_dir=output_dir,
    exp099_cache_path=get_nested(config, 'data.exp099_train_feature_cache_local'),
    exp099_schema_path=get_nested(config, 'data.exp099_train_feature_schema_local'),
    exp114_oof_path=get_nested(config, 'data.exp114_oof_predictions_local'),
    exp114_summary_path=get_nested(config, 'data.exp114_summary_local'),
    max_rows=get_nested(config, 'selector.max_rows'),
)
summary_path = output_dir / 'exp129_spatial_prior_as_selector_candidate_summary.json'
print('summary_path:', summary_path)
print(json.dumps(summary.get('decision', {}), indent=2, sort_keys=True))

## 5. Metrics and artifacts

In [ ]:
metrics_path = output_dir / 'exp129_spatial_prior_as_selector_candidate_metrics.csv'
candidate_path = output_dir / 'exp129_spatial_prior_as_selector_candidate_candidate_metrics.csv'
true_topk_path = output_dir / 'exp129_spatial_prior_as_selector_candidate_true_error_topk_metrics.csv'
pred_topk_path = output_dir / 'exp129_spatial_prior_as_selector_candidate_predicted_topk_metrics.csv'
dist_path = output_dir / 'exp129_spatial_prior_as_selector_candidate_selection_distribution.csv'
bucket_path = output_dir / 'exp129_spatial_prior_as_selector_candidate_bucket_metrics.csv'
importance_path = output_dir / 'exp129_spatial_prior_as_selector_candidate_feature_importance_mean.csv'

metrics = pd.read_csv(metrics_path).sort_values('rmse_tvt')
display(metrics)
display(pd.read_csv(candidate_path))
display(pd.read_csv(true_topk_path))
display(pd.read_csv(pred_topk_path).head(30))
display(pd.read_csv(dist_path).head(50))
display(pd.read_csv(bucket_path).sort_values('rmse_tvt', ascending=False).head(20))
display(pd.read_csv(importance_path).head(30))